# 01 — Stage 1 GRPO (Math: OR1/DAPO/DeepScaler), LoRA

**Requires notebook 00 to have passed every gate and been committed.** LoRA lr=2e-5 is UNVALIDATED (plan §1/§8 item 3) — this is the first run that will produce real evidence either way. If the update-sentinel warns of near-zero relative parameter change, treat that as informative, not as something to silently retune away.

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/WYR186/RLVR.git'  # HTTPS; if the repo is
# private, authenticate interactively (git credential prompt / a token you
# paste when asked) rather than embedding a token in this notebook.
REPO_DIR = '/content/RLVR'
import os
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

import json
from pathlib import Path
CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
print('config loaded:', CONFIG['experiment'])

In [ ]:
audit_confirmed = guru_data.load_confirmed_audit()
splits = json.loads((DATA_DIR / 'exp2_splits.json').read_text())
domain_field = splits['domain_field']
PROMPT_FIELD, ANSWER_FIELD = splits['prompt_field'], splits['answer_field']

raw = guru_data._load_raw(revision=splits['dataset_revision'])
train_split = raw['train'] if 'train' in raw else next(iter(raw.values()))
stage_a_pool = guru_data.filter_stage_subset(train_split, guru_data.STAGE_A_SUBSET_NAMES, domain_field)

stage_a_train = stage_a_pool.select(splits['stage_a_train_idx']).map(lambda ex: {
    'prompt': str(ex[PROMPT_FIELD]), 'answer': str(ex[ANSWER_FIELD])
}, remove_columns=stage_a_pool.column_names)
print('stage-A train rows:', len(stage_a_train))

In [ ]:
RUN_DIR = f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_REPLACE_WITH_HASH/stage_a'
# Replace REPLACE_WITH_HASH with a short content hash of this config once
# decided (matches eaaj-pilot's run-dir convention, src/repro.py:config_hash),
# so this run dir is uniquely identified and resumable.
sa = CONFIG['stage_a']
summary = pipeline.run_stage_a_grpo(
    CONFIG['model_id'], CONFIG['peft'], stage_a_train, RUN_DIR,
    checkpoint_steps=sa['checkpoint_steps'], max_steps=sa['max_steps'],
    learning_rate=sa['learning_rate'], per_device_batch=sa['per_device_train_batch_size'],
    grad_accum=sa['gradient_accumulation_steps'], num_generations=sa['num_generations'],
    beta=sa['beta'], temperature=sa['temperature'], top_p=sa['top_p'],
    max_prompt_length=sa['max_prompt_length'], max_completion_length=sa['max_completion_length'],
    seed=CONFIG['seed'], eval_every=sa['eval_every'])
print(summary)

## Commit reminder

Commit the run directory (dashboard.jsonl, update_sentinel.jsonl, summary.json, ckpt-0/100/200 adapters), prefix `exp2-colab:`. Log wall time + compute-unit cost in `eaaj-pilot/compute_log.md`. If Phase-0's measured s/update implies the de-scope fallback is needed, apply `de_scope_fallback_max_steps` / `de_scope_fallback_checkpoint_steps` from the config and re-run this notebook — do not silently mix a partial 200-update attempt with a restarted 100-update one.